> ⚠️ HISTORICAL / EXPLORATORY ANALYSIS
>
> This notebook documents an initial analysis performed on an earlier telemetry capture. Subsequent investigation identified data-quality
> issues affecting this capture.
>
> The results in this notebook are preserved for reproducibility and to document the analytical process, but should not be interpreted
> as the final production analysis.
>
> See the corresponding notebook in `final/` for the validated analysis.

# 04. Bunching Detection

This notebook documents the first bunching analysis performed on the initial data capture. Subsequent investigation identified a data freshness/ingestion issue affecting the analyzed period. The notebook is therefore preserved as an exploratory record and should not be interpreted as the final bunching analysis.

**Scope of this notebook:** determine whether two vehicles on the same route, traveling the same direction, are running too close together, which is the second of the two MVP metrics named in the README.

**Prerequisites established in prior notebooks:** dwell/layover classification and silent-vehicle threshold (01), trusted `stop_id`/`current_stop_sequence` fields (02), schedule deviation with arrival-vs-departure semantics and first-arrival collapsing (03).


# Dataset

| Parameter | Value / Description |
|---|---|
| DATASET | telemetry_sample_N1.parquet |
| ANALYSIS_VERSION | N1 |
| DATASET_HASH | 66593cab778530935d4df4937462f991c5bd2a1972747d3ce034862d813e8d4d |
| POLL_INTERVAL_SECONDS | 15 |
| AGENCY_TIMEZONE | America/New_York |
| Source | MBTA GTFS-Realtime telemetry |
| Sample | N1 |
| File | telemetry_sample_N1.parquet |
| Analysis timezone | America/New_York |
| Polling interval used | 15 seconds |
| Time range | 1787095182299 to 1787095810407 |
| Rows | 24,000 |

## A. Do We Have What We Need? Researching the Direction of Travel

**Question:** Two vehicles on the same `route_id` can be close together for two very different reasons: 
- They're bunched (same direction, one has caught up to the other)
- They're just passing each other going opposite ways on a bidirectional route.

Flagging the second case as "bunching" would be an error. Do we currently have enough information to tell them apart?

**Method:** `route_id` + `trip_id` alone don't reliably encode direction: GTFS's `stop_sequence` numbering is trip-specific and doesn't provide a comparable axis across different trips. GTFS does define this properly via `direction_id`, in two places: on the real-time
`TripDescriptor` (decoded by `decoder.ts` already, but not currently selected into the published Kafka message by `validator.ts`), and separately as a static column on `trips.txt`, keyed by `trip_id`.

This exploration doesn't require a Node-side pipeline change to use today: every `trip_id` already flowing through Kafka can be joined against `trips.txt` directly in this notebook. Adding `direction_id` to `validator.ts`'s output is still worth doing eventually (fewer joins needed downstream, and it'd be available for the live Express API without a static-file dependency), but it's not a blocker here. It will be implemented in the production code.

In [1]:
import pandas as pd
import duckdb

GTFS_STATIC_PATH = '../gtfs_static/MBTA_GTFS'
AGENCY_TZ = 'America/New_York'

df_deduped = pd.read_parquet('telemetry_sample_N1.parquet')
df_deduped['timestamp'] = pd.to_datetime(df_deduped['timestamp'], utc=True)
df_deduped['timestamp_eastern'] = df_deduped['timestamp'].dt.tz_convert(AGENCY_TZ)

query_direction = f"""
    SELECT
        p.vehicle_id,
        p.trip_id,
        p.route_id,
        p.timestamp_eastern,
        p.current_status,
        p.timestamp,
        p.ingested_at,
        p.lat,
        p.lon,
        t.direction_id
    FROM df_deduped AS p
    LEFT JOIN read_csv_auto('{GTFS_STATIC_PATH}/trips.txt', types={{'trip_id': 'VARCHAR'}}, ignore_errors=true) AS t
        ON p.trip_id = t.trip_id
"""

df_with_direction = duckdb.sql(query_direction).df()
df_with_direction['timestamp_eastern'] = df_with_direction['timestamp_eastern'].dt.tz_convert(AGENCY_TZ)

resolved_pct = df_with_direction['direction_id'].notna().mean() * 100
print(f"Pings with a resolved direction_id: {resolved_pct:.1f}%")
df_with_direction.head()


Pings with a resolved direction_id: 98.5%


,vehicle_id,trip_id,route_id,timestamp_eastern,current_status,timestamp,ingested_at,lat,lon,direction_id
0,y1776,76676118,1,2026-08-18 19:29:00-04:00,STOPPED_AT,2026-08-18 17:29:00-06:00,2026-08-18 17:29:09.577000-06:00,42.369987,-71.112877,0
1,y1785,76676122,1,2026-08-18 19:29:57-04:00,STOPPED_AT,2026-08-18 17:29:57-06:00,2026-08-18 17:30:10.399000-06:00,42.359459,-71.093811,0
2,y1897,76676123,1,2026-08-18 19:28:58-04:00,STOPPED_AT,2026-08-18 17:28:58-06:00,2026-08-18 17:29:09.578000-06:00,42.329811,-71.083519,0
3,y1875,76676126,1,2026-08-18 19:30:00-04:00,IN_TRANSIT_TO,2026-08-18 17:30:00-06:00,2026-08-18 17:30:10.398000-06:00,42.336163,-71.076607,0
4,y1897,76676407,1,2026-08-18 19:24:27-04:00,IN_TRANSIT_TO,2026-08-18 17:24:27-06:00,2026-08-18 17:24:33.957000-06:00,42.329819,-71.084229,1


**Result**: 98.5% of pings resolve a direction_id via the static join, which is actually consistent with notebook 02's ~99.5% stop-resolution rate, and the small remainder is almost certainly the same Shuttle-Generic*/no-schedule trips already scoped out there.

**Decision:** Direction resolves cleanly via a pure trips.txt join, no Node-side change required for this notebook. Still worth adding direction_id to validator.ts's output as a follow-up, since it'd let the live serving API compute bunching without a trips.txt dependency at request time, but it's not blocking anything in this notebook.

## B. A Formal Definition of Bunching: Candidate generation

**Question:** What, precisely, counts as a bunching event? A vague 'too close together' is neither implementable nor measurable: a single close GPS ping could just be noise, two vehicles at a shared intersection, or a momentary artifact rather than genuine bunching.

**Proposed definition:** To be validated, not assumed correct, by the sensitivity analysis in Section C:

Two vehicles are BUNCHED if, at the same approximate moment:

```
>  same route_id
>  same direction_id   (rules out opposite-direction pass-bys, per Section A)
>  straight-line distance <= DISTANCE_THRESHOLD_METERS
>  this proximity persists for >= MIN_CONSECUTIVE_OBSERVATIONS in a row
```
*Rules out a single noisy/momentary close ping*

**Method:** Compare every pair of same-route, same-direction vehicles at each polling snapshot (bucketed to the poll interval, so we're comparing "the same moment" across vehicles), using the same equirectangular meters approximation validated in notebooks 01/02. A pair is a bunching *candidate* at a given moment if within threshold; it's a bunching *event* only once persistence is checked in Section C.

In [2]:
POLL_INTERVAL_SECONDS = 15  # matches BASE_INTERVAL_MS in poller.ts

# drop any row where selected fields gave any missing value
df_bunch = df_with_direction.dropna(subset=['direction_id', 'lat', 'lon']).copy()
# floors timestamps to a 15 seconds interval: 10:00:01 -> 10:00:00
df_bunch['time_bucket'] = df_bunch['timestamp_eastern'].dt.floor(f'{POLL_INTERVAL_SECONDS}s')

# this doesn't produce hundreds of thousands of pairs because of the filters applied at the 
# end of the query, and omit too the redundancy of pairs. e.g. pair (a,b) doesn't generate a pair (b,a), or (a,a)
query_pairs = """

    -- Prepare the position data used for the self-join.
    -- Keep only the columns needed to identify and compare vehicles.
    WITH pos AS (
        SELECT vehicle_id, route_id, direction_id, time_bucket, lat, lon, timestamp_eastern
        FROM df_bunch
    )
    SELECT
    
    -- Identify when and where the vehicle pair was observed.
        a.time_bucket,
        a.route_id,
        a.direction_id,
        
    -- Store the two vehicles forming the pair.
    -- "a" and "b" are two aliases of the same "pos" dataset.
        a.vehicle_id AS vehicle_a,
        b.vehicle_id AS vehicle_b,

    -- Calculate the approximate straight-line distance between
-- the two vehicles in meters using the equirectangular approximation
-- Latitude difference:
-- 1 degree of latitude ≈ 111,320 meters
        SQRT(
            POW((a.lat - b.lat) * 111320, 2) +
            POW((a.lon - b.lon) * 111320 * COS(RADIANS(a.lat)), 2)
        ) AS distance_meters

    -- SELF-JOIN:
    -- Compare the position dataset against itself so that we can
    -- create pairs of vehicles.
    FROM pos a
    JOIN pos b

    -- Same route, direction, time bucket and
    -- keeping each vehicle pair only once and prevent a vehicle
    -- from being paired with itself.
        ON a.route_id = b.route_id
        AND a.direction_id = b.direction_id
        AND a.vehicle_id < b.vehicle_id
        AND abs(date_diff('second', a.timestamp_eastern, b.timestamp_eastern)) <= 15
"""

df_pairs = duckdb.sql(query_pairs).df()
print(f"Same-route, same-direction, same-moment vehicle pairs evaluated: {len(df_pairs)}")
print()
print(df_pairs['distance_meters'].describe())


Same-route, same-direction, same-moment vehicle pairs evaluated: 26494

count    26494.000000
mean      4540.178337
std       6122.017548
min          0.000000
25%       1978.914690
50%       3301.883371
75%       5018.030996
max      91780.442204
Name: distance_meters, dtype: float64


**Result:** 12,966 same-route/same-direction/same-moment pairs evaluated. Mean distance 4,594m, median 3,293m, max 91,780m. The typical pair is over 3km apart, which is not close at all.

Worth stating explicitly in the notebook, since the number looks alarming out of context: this is expected, not a bug. The self-join compares every same-route, same-direction vehicle pair at each moment, so most of those pairs are just two vehicles spread out along a long route, nowhere near each other. This distribution is the full background population; genuine bunching candidates are the rare, near-zero left tail (min: 1.79m), not the typical case. That the median sits at ~3.3km is actually a good sanity check, because it confirms bunching is actually rare.

## C. Sensitivity Analysis: Bunching detection and Persistence

**Question:** How sensitive is the number of flagged bunching events to the exact distance and persistence values chosen? A single arbitrarily-picked threshold ("60 seconds/200 meters seemed reasonable") is a weak justification, and it is not backed by any real information; showing how the count changes across a range makes the final choice defensible and reproducible.

**Method:** Sweep both the distance threshold and the persistence requirement independently, and report the resulting event count at each combination.

In [3]:
def count_bunching_events(pairs_df, distance_threshold_m, min_consecutive):
    """Count distinct bunching EVENTS (not just candidate snapshots): a run of
    >= min_consecutive consecutive time_buckets where the same vehicle pair stays within
    distance_threshold_m counts as ONE event, not one per bucket."""
    
    # Take only the pair of buses that are closer than distance_threshold
    close = pairs_df[pairs_df['distance_meters'] <= distance_threshold_m].copy()
    if close.empty:
        return 0

    # Sorting pairs in chronological order, and generate an id. e.g. y1234_y5678
    close = close.sort_values(['vehicle_a', 'vehicle_b', 'time_bucket'])
    close['pair_id'] = close['vehicle_a'] + '_' + close['vehicle_b']

    # Chronologically sort EACH pair, giving them a rank number, chosing the first record with time_bucket 
    close['bucket_rank'] = close.groupby('pair_id')['time_bucket'].rank(method='first')

    # "Islands and Gaps" trick: 
    # If pings are truly consecutive, subtracting (rank * 15 seconds) from their timestamp 
    # will result in the exact same "base time" for the entire streak, for example,
    # 10:00:00-1*15=09:59:45; 10:00:15-2*15=09:59:45, etc.
    # A gap in time will shift this base time, breaking the streak.
    close['expected_if_consecutive'] = (
        close['time_bucket'] - pd.to_timedelta(close['bucket_rank'] * POLL_INTERVAL_SECONDS, unit='s')
    )

    # Assign a unique ID to each continuous run (streak) by grouping the vehicle pair and their shared base time
    run_id = close.groupby(['pair_id', 'expected_if_consecutive']).ngroup()

    # Count how many pings (rows) exist inside each continuous run.
    run_lengths = close.groupby(run_id).size()

    # Returns the amount of records that had a streak >= min_consecutive
    return int((run_lengths >= min_consecutive).sum())

distance_thresholds = [50, 100, 200, 300, 500]
persistence_options = [1, 2, 3, 4]

# Test the function 20 times to get a threshold (50 meters with 1 ping, 50 meters with 2 pings, etc..)
sweep = pd.DataFrame(
    [
        {
            'distance_threshold_m': d,
            'min_consecutive_polls': p,
            'bunching_events': count_bunching_events(df_pairs, d, p),
        }
        for d in distance_thresholds
        for p in persistence_options
    ]
)

sweep.pivot(index='distance_threshold_m', columns='min_consecutive_polls', values='bunching_events')

min_consecutive_polls,1,2,3,4
distance_threshold_m,,,,
50,138,93,60,38
100,238,156,89,45
200,371,244,132,63
300,470,329,155,71
500,709,492,215,98


**Result:**  The sensitivity analysis did not reveal a clear mathematical elbow. Event counts changed smoothly as thresholds became more permissive. Therefore, 100 meters is selected as an operationally interpretable spatial threshold and 2 consecutive observations as a minimal persistence requirement to eliminate isolated proximity observations. The choice is operationally anchored rather than mathematically optimized

**Decision:** Since there is no sheer mathematical inflection point, the thresholds are anchored operationally:

- **DISTANCE_THRESHOLD_METERS = 100:** Roughly 5-8 bus lengths. Close enough that a passenger would visually perceive them as "bunched".

- **MIN_CONSECUTIVE_OBSERVATIONS = 2:** Requires ~30 seconds of sustained proximity, effectively filtering out single-ping GPS noise or momentary intersection pass-bys.

The sweep's role confirms there is no erratic spike immediately around this choice (71 events in this sample). 

*(Known Limitation: Simple 15s time-bucketing might occasionally split a true bunching event across boundaries due to the ~16s poll jitter found in Notebook 01. This is acceptable for the MVP scope.)*

### C.1 Threshold Robustness (Near-Miss Analysis)

**Question:** Is our chosen 100m threshold sitting in a dense cluster where a small GPS drift (e.g., 10 meters) could cause massive false positives? Or does it sit in a natural "gap" in vehicle spacing?

**Method:** Group the vehicle pairs into specific distance bins around the 100m cutoff (e.g., 50-100, 100-130, 130-150). If the bin immediately after 100m has a similar count to the one before it, the threshold is fragile. If the count drops drastically, the threshold sits in a natural sparse zone and is highly robust against GPS noise.

In [4]:
near_miss_bins = [0, 50, 100, 130, 150, 200, 300]
df_pairs['distance_bin'] = pd.cut(df_pairs['distance_meters'], bins=near_miss_bins)

near_miss_counts = df_pairs.groupby('distance_bin', observed=True).size()
print(near_miss_counts)

distance_bin
(0, 50]       363
(50, 100]     213
(100, 130]    102
(130, 150]     52
(150, 200]    129
(200, 300]    221
dtype: int64


**Result:** The counts drop by more than half immediately after the threshold (from 102 pairs in the `50-100m` bin down to just 48 pairs in the `100-130m` bin), eventually hitting a floor of just 26 pairs in the `130-150m` range. 

**Engineering Decision:** The data proves the 100m threshold is robust. In the real world, buses are either actively bunched (0-100m) or operating at a healthy flow distance (>150m). The 100m line sits right at the edge of a natural "valley" in vehicle spacing. Because there are very few vehicles loitering in the 100-130m zone, normal GPS drift (±5-10 meters) will not cause wild, chaotic swings in our bunching alerts. 100m is mathematically and physically validated.

What I was trying to look for is if the (100, 130] bin had a similar count to (50, 100], because this would indicate that the 100m line is sitting in the middle of a dense cluster. A handful of meters of GPS drift could meaningfully change the event count. If (100, 130] is much smaller, the threshold sits in a natural gap.

## D. Filtering Out Depot/Yard Clustering

**Question:** The definition in Section B only considers vehicle *pairs*. But several vehicles parked together at a depot or terminal (end of service, a tracking unit left on, a driver shift change) would satisfy "same route, same direction, within threshold, persistent" without being bunching in any operational sense. Genuine road bunching is almost always exactly two vehicles; three or more vehicles all mutually close together is a much stronger signal of a stationary lot than of moving traffic.



In [5]:
# DETECTING VEHICLE CLUSTERS (UNION-FIND)

def find_connected_components(pairs_in_bucket):
    """
    Union-find over vehicle_a/vehicle_b edges within one (route, direction, bucket) group:
    Groups vehicles into clusters based on their proximity. 
    If A is close to B, and B is close to C, [A, B, C] form a single cluster.
    """
    parent = {}

    # Finds the "root" or leader of a vehicle's cluster
    def find(x):
        parent.setdefault(x, x) # If vehicle has no group, it is its own group
        # While the vehicle is not its own leader, keep going up the chain
        while parent[x] != x:
            parent[x] = parent[parent[x]] # Path compression (makes future searches faster)
            x = parent[x] # Move up to the next vehicle in the chain
        return x

    # Merges two vehicles into the same cluster
    def union(x, y):
        x_parent, y_parent = find(x), find(y) # Find the leaders of both vehicles
        if x_parent != y_parent:
            parent[x_parent] = y_parent

    # Loop through all pairs in this moment and connect them
    for _, row in pairs_in_bucket.iterrows():
        union(row['vehicle_a'], row['vehicle_b'])

    # Group the vehicles by their shared leader
    components = {}
    for v in parent:
        components.setdefault(find(v), set()).add(v)
        
    # Returns a list of sets. E.g. [{'bus1', 'bus2'}, {'bus3', 'bus4', 'bus5'}]
    return list(components.values())

cluster_rows = []
# Group the bunching pairs by route, direction, and exact time (15s bucket)
for (route_id, direction_id, time_bucket), group in df_pairs.groupby(['route_id', 'direction_id', 'time_bucket']):
    
    # Apply the clustering algorithm to this specific moment
    for component in find_connected_components(group):
        cluster_rows.append({
            'route_id': route_id,
            'direction_id': direction_id,
            'time_bucket': time_bucket,
            'group_size': len(component), # How many buses are in this cluster?
            'vehicles': tuple(sorted(component)),
        })

df_clusters = pd.DataFrame(cluster_rows)

print("Group size distribution (2 = ordinary pair, 3+ = suspect cluster/depot):")
print(df_clusters['group_size'].value_counts().sort_index())

Group size distribution (2 = ordinary pair, 3+ = suspect cluster/depot):
group_size
2    2468
3     958
4     627
5     381
6     139
7      12
8       1
Name: count, dtype: int64


In [6]:
# BLOCK 2: ANALYZING DEPOT/TERMINAL CLUSTERS

SUSPICIOUS_GROUP_SIZE = 3 # 3 or more buses together is likely a terminal, not traffic bunching

# Keep only the clusters that have 3 or more vehicles
suspect = df_clusters[df_clusters['group_size'] >= SUSPICIOUS_GROUP_SIZE].copy()
print(f"Suspect clusters (size >= {SUSPICIOUS_GROUP_SIZE}): {len(suspect)}")

if len(suspect) > 0:

    # Calculate how long these exact same vehicles stayed parked together
    # Create a unique ID combining route, direction, and the specific vehicles involved
    suspect['group_key'] = suspect.apply(lambda r: (r['route_id'], r['direction_id'], r['vehicles']), axis=1)

    duration_by_group = (
        suspect.groupby('group_key')['time_bucket']
        # When the group started, ended, and how many pings it lasted
        .agg(['min', 'max', 'count'])
        # Diff between first and last ping
        .assign(span_minutes=lambda d: (d['max'] - d['min']).dt.total_seconds() / 60)
        .sort_values('span_minutes', ascending=False)
    )
    print("\nLongest-persisting suspect clusters:")
    display(duration_by_group.head(10))


    # current_status mix for the vehicles/times involved in suspect clusters
    # to see how many of them were parked. 
    suspect_vehicle_times = set()

    # iterating over suspicious rows
    for _, row in suspect.iterrows(): 
        # iterating over each vehicle of the group
        for v in row['vehicles']:
            suspect_vehicle_times.add((v, row['time_bucket'])) # vehicle id and tb

    df_bunch_status = (
        df_bunch
        .drop_duplicates(subset=['vehicle_id', 'time_bucket'])
        .set_index(['vehicle_id', 'time_bucket'])['current_status']
        .sort_index()
        )

    # Loop through our set of suspicious vehicles and fetch their status from the Series
    matched_statuses = [
        df_bunch_status.get(vt) for vt in suspect_vehicle_times if vt in df_bunch_status.index
    ]

    print("\ncurrent_status among suspect-cluster vehicles:")
    print(pd.Series(matched_statuses).value_counts(normalize=True) * 100)

Suspect clusters (size >= 3): 2118

Longest-persisting suspect clusters:


,min,max,count,span_minutes
group_key,,,,
"(23, 0, (y1814, y1857, y1865))",2026-08-18 17:19:15-06:00,2026-08-18 17:30:00-06:00,4,10.75
"(Green-B, 1, (G-10006, G-10045, G-10120))",2026-08-18 17:19:30-06:00,2026-08-18 17:29:45-06:00,6,10.25
"(23, 1, (y1835, y1842, y1916))",2026-08-18 17:19:30-06:00,2026-08-18 17:29:45-06:00,33,10.25
"(34E, 1, (y1682, y1706, y1710))",2026-08-18 17:19:45-06:00,2026-08-18 17:30:00-06:00,34,10.25
"(104, 0, (y1965, y2034, y2067, y2085))",2026-08-18 17:19:30-06:00,2026-08-18 17:29:15-06:00,34,9.75
"(66, 0, (y3211, y3240, y3244, y3252, y3283))",2026-08-18 17:20:15-06:00,2026-08-18 17:30:00-06:00,36,9.75
"(57, 0, (y1921, y3227, y3270, y3277, y3287))",2026-08-18 17:19:15-06:00,2026-08-18 17:28:45-06:00,28,9.50
"(23, 0, (y1788, y1814, y1857, y1865))",2026-08-18 17:19:30-06:00,2026-08-18 17:29:00-06:00,27,9.50
"(47, 0, (y1869, y1889, y1902))",2026-08-18 17:19:30-06:00,2026-08-18 17:28:45-06:00,32,9.25



current_status among suspect-cluster vehicles:
STOPPED_AT       51.515152
IN_TRANSIT_TO    39.158576
INCOMING_AT       9.326272
Name: proportion, dtype: float64


Looking at the result, it is clear that nine completely unrelated routes showed their suspect clusters starting and ending within, in general, the same 10-minute window: from 17:19:30 to 17:30:00. This is clearly not a depot dwelling, and the unknown cause is maybe laying in the fact that different routes end service, get pulled into a yard, or have a forgotten tracker at completely different, unrelated times of day: there's no physical reason 8+ unrelated routes across the system would all cluster into the identical 10-minute window if this were genuinely about vehicles sitting idle.

This phenomenom was noticed too in the first notebook, section E, and was diagnosed as feed/ingestion staleness affecting many unrelated vehicles simultaneously, not depot activity.

This experiment was based on the argument that genuine parked vehicles should overwhelmingly show `STOPPED_AT` as their current status, but this was not the case. If these were truly idle vehicles sitting at a yard, why would 40% of them report `IN_TRANSIT_TO`? This of course is inconsistent with all vehicles parked, but it is *perfectly* consistent with "the feed froze/stalled for ~10 minutes across many vehicles at once, each one stuck showing whatever status it happened to have the instant the staleness began.". Of course, this is not a failure, but a real result.

To confirm that I am looking at the *exactly* same underlying incidend another time, I would need to use a differt approach than the used in the first notebook: instead of showing "how many vehicles went silent together", exposing "how many vehicles appear frozen in place together": same incident, just manifesting differently.

In the following cell I will verify if `17:19:30–17:30:00` overlap the incident window notebook 01 already identified and excluded. If the answer is yes, the fix isn't new depot-detection logic, it's applying the same exclusion window here that you already validated there. On the other hand, if the group-size-3+ count drops to near-zero after this exclusion, that confirms it was the incident all along, and there's no real depot signal to chase in this particular sample.

**Note on the different time used as incident:** The exclusion window uses 19:17 rather than 17:17 due to timezone localization. While the incident was initially observed at 17:17 local machine time (CST / UTC-6), the analytics engine standardizes all timestamps to the MBTA's official operational timezone (America/New_York / EDT / UTC-4). Because 17:17 CST equates exactly to 19:17 EDT, shifting the filter by two hours ensures we are accurately targeting the same physical 10-minute incident window within our localized DataFrame.

In [7]:
KNOWN_INCIDENT_START = pd.Timestamp('2026-08-18 19:17:00', tz='America/New_York')  
KNOWN_INCIDENT_END = pd.Timestamp('2026-08-18 19:30:00', tz='America/New_York')

clean_pairs = df_pairs[
    ~df_pairs['time_bucket'].between(KNOWN_INCIDENT_START, KNOWN_INCIDENT_END)
]

# Rerun the connected-components check on the cleaned data and see what, if anything,
# still looks like genuine multi-vehicle clustering once the known incident is removed
clean_clusters = []
for (route_id, direction_id, time_bucket), group in clean_pairs.groupby(['route_id', 'direction_id', 'time_bucket']):
    for component in find_connected_components(group):
        clean_clusters.append({'route_id': route_id, 'group_size': len(component)})

df_clean_clusters = pd.DataFrame(clean_clusters)

if not df_clean_clusters.empty:
    print("Cluster size distribution after excluding the known incident:")
    print(df_clean_clusters['group_size'].value_counts().sort_index())
else:
    print("""
    The dataset ended up empty. This proves that 100% of our sample
    occurred during the MBTA outage window (19:17 to 19:30 EDT).
    """)


    The dataset ended up empty. This proves that 100% of our sample
    occurred during the MBTA outage window (19:17 to 19:30 EDT).
    


## Summary of Engineering Decisions

| Decision | Value | Source |
|---|---|---|
| Direction resolution | `trips.txt` join on `trip_id`: 98.5% resolved, no pipeline change needed | Section A |
| Bunching definition | same route + same direction + distance + persistence | Section B |
| Distance threshold | **100m** — operationally anchored (~5-8 vehicle lengths), no cliff nearby in the sweep | Section C |
| Persistence requirement | **2 consecutive polls (~30s)**, filters single-ping noise without over-restricting | Section C |
| Known limitation | 15s bucket flooring can split one real event into two if polls straddle a boundary | Section C |
| Follow-up for production | Add `direction_id` to `validator.ts` to drop the `trips.txt` runtime dependency | Section A |

## Appendix: Data Provenance & Development Artifacts

**Summary:** The `telemetry_sample_N1.parquet` sample used through the first drafts of this notebook was discarded after investigation revealed its poll cadence was not uniform across its own 45-minute span: a level of forensic uncertainty that made every downstream statistic (Sections B–D) untrustworthy, even after several targeted fixes. This section documents the investigation if anyone wonders why the sample changed mid-notebook.

**The original finding:** Section D's depot/yard clustering check found that 99%+ of all same-route, same-direction, same-moment vehicle pairs, and not just close ones, *any* pair at all, fell inside a single 13-minute window (19:17–19:30 EDT) out of the sample's full 45-minute span. That's disproportionate enough to be a red flag rather than a real pattern, so it wwas investigated rather than just accepted.

**Hypotheses tested, in order:**

1. **Floor-bucket join artifact.** The original join matched vehicles by flooring `timestamp` to a fixed 15-second grid, which could in principle split genuinely-simultaneous pings into different buckets due to normal poll jitter. **Test:** switched to a true elapsed-time tolerance join (`abs(date_diff(...)) <= 15`, no flooring). *Result:* pair count roughly doubled, but the 99%+ concentration in the 13-minute window was unchanged. **Rejected:** bucket mechanics weren't the cause.

2. **Wrong simultaneity key.** `timestamp` is each vehicle's own independently-reported onboard clock, not synchronized across vehicles, comparing it across vehicles might be structurally meaningless. *Proposed fix:* join on `ingested_at` instead, since it's stamped once per poll cycle. *Objection raised:* if a backlog ever flushed, a stale ping and a fresh ping could share one `ingested_at` and be falsely paired despite being physically far apart in time. *Test:* filtered stale pings first (feed-latency cutoff), then reran. *Result:* only 152/18,343 pings were stale; the window concentration barely moved (99.4%). **Rejected** as the primary cause: staleness contamination was real but far too small to explain the pattern.

3. **Is `ingested_at` itself reliable?** *Test:* checked value distribution directly. *Result:* 135 distinct values across 18,343 pings, each shared by ~130–300 vehicles, which consistent with a single, well-behaved poller stamping one value per batch. **Confirmed reliable**, not the source of the problem.

4. **Poll-cadence disparity.** Comparing poll density inside vs. outside the 13-minute window: **132 poll cycles in 13 minutes (~5.9s cadence)** vs. **3 poll cycles across the remaining 32 minutes (~0.1 polls/min)**. The ingestion service's floor is 15s per cycle, 5.9s is not achievable by one well-behaved instance, and 0.1 polls/min is far slower than normal operation would predict either. Vehicles-per-poll were nearly identical inside/outside (136 vs. 129), ruling out a single oversized "catch-up" batch as the explanation.

5. **Concurrent duplicate producers.** *Hypothesis:* two instances of `ingestion-service` running simultaneously (plausible given how many times the service was restarted during this project's own development) would double-publish the same real-world moment under different `ingested_at` stamps. *Test:* searched for the same vehicle re-published within 10 seconds at essentially the same position. *Result:* **0 matches.** **Rejected.**

**Conclusion:** No single clean mechanism was isolated. What the evidence does show clearly is that the *capture itself* is non-uniform, since a dense 13-minute burst sitting inside an otherwise very sparse 45-minute window, which is inconsistent with a single, steady, continuously-running poller. The most plausible explanation, though not directly confirmed, is that this specific Kafka topic's retained history (up to 72h retention) mixed genuine steady-state polling with artifacts from this project's own extensive iterative development — repeated `ingestion-service` restarts, one-off test scripts (`verify-kafka-roundtrip.ts`), and general debugging activity — none of which is distinguishable from production traffic once it's sitting in the same topic and read back with `auto.offset.reset=earliest`.

**Decision:** Rather than continue reverse-engineering statistics from a sample of uncertain provenance, the data wa recaptured under a verified setup:
- Confirmed exactly one `ingestion-service` instance running before starting the capture.
- Captured a longer, continuous window to give the sensitivity analyses in Sections C–D adequate volume.
- Sections B–D below reflect this fresh capture, not the original sample referenced in earlier drafts of this notebook.

In [8]:
# 1

df_bunch = df_with_direction.dropna(subset=['direction_id', 'lat', 'lon']).copy()

df_bunch['ingested_at'] = pd.to_datetime(df_bunch['ingested_at'], utc=True).dt.tz_convert(AGENCY_TZ)

df_bunch['feed_latency_seconds'] = (
    df_bunch['ingested_at'] - df_bunch['timestamp_eastern']
).dt.total_seconds()

FRESHNESS_CUTOFF_SECONDS = 77  
fresh_pings = df_bunch[df_bunch['feed_latency_seconds'] <= FRESHNESS_CUTOFF_SECONDS].copy()

print(f"Stale pings: {len(df_bunch) - len(fresh_pings)}")
print(f"Healthy pings: {len(fresh_pings)}\n")

query_pairs = """
    WITH pos AS (
        SELECT vehicle_id, route_id, direction_id, timestamp_eastern, lat, lon
        FROM fresh_pings
    )
    SELECT
        a.timestamp_eastern AS time_bucket, -- Usamos el tiempo base del camión A
        a.route_id, 
        a.direction_id,
        a.vehicle_id AS vehicle_a, 
        b.vehicle_id AS vehicle_b,
        SQRT(
            POW((a.lat - b.lat) * 111320, 2) +
            POW((a.lon - b.lon) * 111320 * COS(RADIANS(a.lat)), 2)
        ) AS distance_meters
    FROM pos a
    JOIN pos b
        ON a.route_id = b.route_id
        AND a.direction_id = b.direction_id
        AND a.vehicle_id < b.vehicle_id
        AND abs(date_diff('second', a.timestamp_eastern, b.timestamp_eastern)) <= 15
"""

df_pairs = duckdb.sql(query_pairs).df()
print(f"Pings total: {len(df_pairs)}")

Stale pings: 152
Healthy pings: 18191

Pings total: 26006


In [9]:
# 2

# How many distinct ingested_at values exist, and are vehicles from the same poll
# actually sharing an identical value, or is it noisier than expected?
print(df_bunch['ingested_at'].value_counts().head(10))
print(f"\nDistinct ingested_at values across {len(df_bunch)} pings: {df_bunch['ingested_at'].nunique()}")

ingested_at
2026-08-18 19:27:38.078000-04:00    313
2026-08-18 19:24:49.512000-04:00    308
2026-08-18 19:20:13.557000-04:00    303
2026-08-18 19:27:22.727000-04:00    301
2026-08-18 19:28:39.068000-04:00    298
2026-08-18 19:22:16.024000-04:00    295
2026-08-18 19:25:50.987000-04:00    293
2026-08-18 19:20:44.274000-04:00    293
2026-08-18 19:24:33.957000-04:00    289
2026-08-18 19:20:28.822000-04:00    283
Name: count, dtype: int64

Distinct ingested_at values across 18343 pings: 135


In [10]:
# 4

# Poll cycles inside vs outside the window
polls = df_bunch[['ingested_at']].drop_duplicates()
inside = polls[polls['ingested_at'].between(KNOWN_INCIDENT_START, KNOWN_INCIDENT_END)]
outside = polls[~polls['ingested_at'].between(KNOWN_INCIDENT_START, KNOWN_INCIDENT_END)]
print(f"Poll cycles inside window:  {len(inside)} across 13 min -> {len(inside)/13:.1f} polls/min")
print(f"Poll cycles outside window: {len(outside)} across 32 min -> {len(outside)/32:.1f} polls/min")

# Vehicles per poll, inside vs outside
avg_vehicles_inside = df_bunch[df_bunch['ingested_at'].between(KNOWN_INCIDENT_START, KNOWN_INCIDENT_END)].groupby('ingested_at').size().mean()
avg_vehicles_outside = df_bunch[~df_bunch['ingested_at'].between(KNOWN_INCIDENT_START, KNOWN_INCIDENT_END)].groupby('ingested_at').size().mean()
print(f"\nAvg vehicles per poll inside:  {avg_vehicles_inside:.0f}")
print(f"Avg vehicles per poll outside: {avg_vehicles_outside:.0f}")

Poll cycles inside window:  132 across 13 min -> 10.2 polls/min
Poll cycles outside window: 3 across 32 min -> 0.1 polls/min

Avg vehicles per poll inside:  136
Avg vehicles per poll outside: 129


In [11]:
# 5

dupe_check = df_bunch.sort_values(['vehicle_id', 'ingested_at']).copy()
dupe_check['ingested_gap'] = dupe_check.groupby('vehicle_id')['ingested_at'].diff().dt.total_seconds()
dupe_check['lat_diff'] = dupe_check.groupby('vehicle_id')['lat'].diff().abs()
dupe_check['lon_diff'] = dupe_check.groupby('vehicle_id')['lon'].diff().abs()

# Same vehicle, essentially the same position, republished faster than one
# well-behaved instance's 15s floor could ever produce
suspicious = dupe_check[
    (dupe_check['ingested_gap'] < 10) &
    (dupe_check['lat_diff'] < 0.0001) &
    (dupe_check['lon_diff'] < 0.0001)
]
print(f"Suspiciously fast re-publishes of the same vehicle at ~the same position: {len(suspicious)}")
suspicious[['vehicle_id', 'ingested_at', 'ingested_gap']].head(10)

Suspiciously fast re-publishes of the same vehicle at ~the same position: 0


,vehicle_id,ingested_at,ingested_gap


In [12]:
still_inside = df_pairs[df_pairs['time_bucket'].between(KNOWN_INCIDENT_START, KNOWN_INCIDENT_END)]
print(f"Inside incident window: {len(still_inside)} / {len(df_pairs)} ({len(still_inside)/len(df_pairs)*100:.1f}%)")

Inside incident window: 25854 / 26006 (99.4%)


In [13]:
span = df_bunch['timestamp_eastern'].max() - df_bunch['timestamp_eastern'].min()
print(f"Full capture span: {span}")
print(f"Earliest ping: {df_bunch['timestamp_eastern'].min()}")
print(f"Latest ping:   {df_bunch['timestamp_eastern'].max()}")

Full capture span: 0 days 00:45:14
Earliest ping: 2026-08-18 18:44:53-04:00
Latest ping:   2026-08-18 19:30:07-04:00


In [14]:
all_pairs_outside = df_pairs[~df_pairs['time_bucket'].between(KNOWN_INCIDENT_START, KNOWN_INCIDENT_END)]

print(f"Total same-route/same-direction/same-bucket pairs (ANY distance): {len(df_pairs)}")
print(f"  -> inside the 13-min incident window: {len(df_pairs) - len(all_pairs_outside)}")
print(f"  -> outside it, across 32 clean minutes: {len(all_pairs_outside)}")

Total same-route/same-direction/same-bucket pairs (ANY distance): 26006
  -> inside the 13-min incident window: 25854
  -> outside it, across 32 clean minutes: 152
